In [4]:
# calculate tencent TAR 项目根目录新建test和wrong/tencent目录哦
import numpy as np
import torch
import torchvision
from tqdm import tqdm
from official_api.tencent import face_compare

from utils.dataset import Dataset

print("------------------------加载测试集-------------------------")
test_set = Dataset("C:/yy/datasets/lfw/lfw-aligned-112x112/")
print("-----------------------启动分批队列------------------------")
batch_size = 8
test_set.start_batch_queue(
    batch_size=batch_size,
    batch_format="random_samples",
    transforms=torchvision.transforms.Compose([
        torchvision.transforms.Resize((112,112)),  # 调整图片大小
        torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
        torchvision.transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # 归一化
    ]),
    num_threads=3
)

print("-----------------------开始api测试------------------------")
with torch.no_grad():
    epoch = 750
    before = []
    x=0
    for i in tqdm(range(0, epoch), total=epoch):
        batch = test_set.pop_batch_queue()
        source_faces = batch['sources'].cuda()
        sources2_faces = batch['sources2'].cuda()
        for j in range(0, batch_size):
            torchvision.utils.save_image([source_faces[j] * 0.5 + 0.5], './test/source.png', nrow=1)
            torchvision.utils.save_image([sources2_faces[j] * 0.5 + 0.5], './test/sources.png', nrow=1)
            res = face_compare(face1_path='./test/source.png', face2_path='./test/sources.png')
            if res is None:
                x = x+1
                torchvision.utils.save_image([source_faces[j] * 0.5 + 0.5], './wrong/tencent/source{}_{}.png'.format(x, batch['sources_name'][j]), nrow=1)
                torchvision.utils.save_image([sources2_faces[j] * 0.5 + 0.5], './wrong/tencent/source2{}_{}.png'.format(x, batch['sources_name'][j]), nrow=1)
            else:
                before.append(res)
    print(np.mean(before))
    before = np.array(before)
    print(np.sum(before > 40) / len(before))
    print(np.sum(before > 50) / len(before))
    print(np.sum(before > 60) / len(before))


------------------------加载测试集-------------------------
7606 images of 901 classes loaded
-----------------------启动分批队列------------------------
-----------------------开始api测试------------------------


  1%|          | 6/750 [00:26<54:59,  4.43s/it]


KeyboardInterrupt: 

In [5]:
# calculate tencent FAR
import numpy as np
import torch
import torchvision
from tqdm import tqdm
from official_api.tencent import face_compare

from utils.dataset import Dataset

print("------------------------加载测试集-------------------------")
test_set = Dataset("C:/yy/datasets/lfw/lfw-aligned-112x112/")
print("-----------------------启动分批队列------------------------")
batch_size = 8
test_set.start_batch_queue(
    batch_size=batch_size,
    batch_format="random_samples",
    transforms=torchvision.transforms.Compose([
        torchvision.transforms.Resize((112,112)),  # 调整图片大小
        torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
        torchvision.transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # 归一化
    ]),
    num_threads=3,
)

print("-----------------------开始api测试------------------------")
with torch.no_grad():
    epoch = 750
    before = []
    x=0
    for i in tqdm(range(0, epoch), total=epoch):
        batch = test_set.pop_batch_queue()
        source_faces = batch['sources'].cuda()
        target_faces = batch['targets'].cuda()
        for j in range(0, batch_size):
            torchvision.utils.save_image([source_faces[j] * 0.5 + 0.5], './test/source.png', nrow=1)
            torchvision.utils.save_image([target_faces[j] * 0.5 + 0.5], './test/target.png', nrow=1)
            res = face_compare(face1_path='./test/target.png', face2_path='./test/source.png')
            if res is None:
                x = x+1
                torchvision.utils.save_image([source_faces[j] * 0.5 + 0.5], './wrong/tencent/source{}_{}.png'.format(x, batch['sources_name'][j]), nrow=1)
                torchvision.utils.save_image([target_faces[j] * 0.5 + 0.5], './wrong/tencent/target{}_{}.png'.format(x, batch['targets_name'][j]), nrow=1)
            else:
                before.append(res)
    print(np.mean(before))
    before = np.array(before)
    print(np.sum(before < 40) / len(before))
    print(np.sum(before < 50) / len(before))
    print(np.sum(before < 60) / len(before))

------------------------加载测试集-------------------------
7606 images of 901 classes loaded
-----------------------启动分批队列------------------------
-----------------------开始api测试------------------------


  0%|          | 2/750 [00:08<55:14,  4.43s/it]


KeyboardInterrupt: 

In [6]:
# test your pretrained model with tencent API
import numpy as np
import torch
import torchvision
from tqdm import tqdm
from official_api.tencent import face_compare

from AdvFaceGAN import Generator
from utils.dataset import Dataset

print("------------------------加载测试集-------------------------")
test_set = Dataset("C:/yy/datasets/lfw/lfw-aligned-112x112/", "target")
print("-----------------------启动分批队列------------------------")
batch_size = 8
test_set.start_batch_queue(
    batch_size=batch_size,
    batch_format="random_samples",
    transforms=torchvision.transforms.Compose([
        torchvision.transforms.Resize((112,112)),  # 调整图片大小
        torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
        torchvision.transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # 归一化
    ]),
    num_threads=3,
)
print("-----------------------加载生成器------------------------")
fake_generator = Generator(is_target=True).eval().cuda()
model_generator_dict = torch.load(r"save_dir\target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15\model\02490_generator.pth", weights_only=True)
fake_generator.load_state_dict(model_generator_dict)

print("-----------------------开始api测试------------------------")
with torch.no_grad():
    epoch = 750
    after = []
    x=0
    for i in tqdm(range(0, epoch), total=epoch):
        batch = test_set.pop_batch_queue()
        source_faces = batch['sources'].cuda()
        target_faces = batch['targets'].cuda()
        perts, fake_afters = fake_generator.forward(source_faces, target_faces)
        for j in range(0, batch_size):
            torchvision.utils.save_image([target_faces[j] * 0.5 + 0.5], './test/target.png', nrow=1)
            torchvision.utils.save_image([fake_afters[j] * 0.5 + 0.5], './test/fake.png', nrow=1)
            res = face_compare(face1_path='./test/target.png', face2_path='./test/fake.png')
            if res is None:
                x = x+1
                torchvision.utils.save_image([source_faces[j] * 0.5 + 0.5], './wrong/tencent/source{}_{}.png'.format(x, batch['sources_name'][j]), nrow=1)
                torchvision.utils.save_image([target_faces[j] * 0.5 + 0.5], './wrong/tencent/target{}_{}.png'.format(x, batch['targets_name'][j]), nrow=1)
            else:
                after.append(res)
    
    print(np.mean(after))
    after = np.array(after)
    print(np.sum(after > 40) / len(after))
    print(np.sum(after > 50) / len(after))
    print(np.sum(after > 60) / len(after))



------------------------加载测试集-------------------------
7606 images of 901 classes loaded
-----------------------启动分批队列------------------------
-----------------------加载生成器------------------------
-----------------------开始api测试------------------------


  0%|          | 1/750 [00:06<1:22:28,  6.61s/it]


KeyboardInterrupt: 